# ViTASA — Module Tuning (vòng 2)

Vòng ablation đầu (10 epoch, thông số module MẶC ĐỊNH) cho thấy 2 module đang
LÀM GIẢM điểm ở phần lớn config, tệ nhất ở restaurant. Notebook này thử 3 giả
thuyết bằng cách chỉnh thông số module, xem có đảo ngược được không:

| Biến thể | Test giả thuyết |
|---|---|
| `focal_g1` (gamma 1.0) | Focal gamma 2.0 quá cao → overcorrect ở domain mất cân bằng nặng |
| `focal_noalpha` (focal thuần) | Focal + class-weight chồng lấn (2 cơ chế cùng đẩy về minority) |
| `norm_keepexpr` | Text Normalization đang XOÁ nhầm marker cảm xúc (kk/haha/!!!) |

**Resume-safe + tự backup Drive** giống notebook chính — chạy được bao nhiêu hay
bấy nhiêu, hết quota mở lại chạy tiếp không mất phần đã xong.

Thứ tự chạy được xếp: **mobile trước** (rẻ nhất ~20p, lấy tín hiệu nhanh) →
restaurant (ca tệ nhất, quan trọng nhất) → hotel.

In [ ]:
# 1. Clone dataset
!git clone https://github.com/kh4nh12/ViTASA.git ViTASA_repo 2>&1 | grep -E '(Cloning|done)'
!ls -lh ViTASA_repo/*.jsonl

In [ ]:
# 2. Install deps
!pip install -q torch transformers scikit-learn seqeval underthesea
import torch
print(f"✅ PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 3. Setup dataset folders
import os, shutil
os.makedirs('VITASA_Enhanced/baseline/data', exist_ok=True)
for domain in ['mobile', 'restaurant', 'hotel']:
    os.makedirs(f'VITASA_Enhanced/baseline/data/{domain}', exist_ok=True)
    src, dst = f'ViTASA_repo/{domain}.jsonl', f'VITASA_Enhanced/baseline/data/{domain}/{domain}.jsonl'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✅ {domain}: {sum(1 for _ in open(dst))} samples")

In [ ]:
# 4. Lấy code mới nhất từ GitHub (cần đã git push train_pair.py bản có --focal-gamma/--focal-no-alpha)
import os
REPO_URL = "https://github.com/Hunganh1305/VITASA_Enhanced.git"
if os.path.isdir("VITASA_Enhanced/.git"):
    !cd VITASA_Enhanced && git pull
else:
    !rm -rf VITASA_Enhanced_code_tmp
    !git clone {REPO_URL} VITASA_Enhanced_code_tmp
    !rsync -a VITASA_Enhanced_code_tmp/ VITASA_Enhanced/ --exclude 'baseline/data'
    !rm -rf VITASA_Enhanced_code_tmp
# Kiểm tra code mới đã có 2 flag chưa (nếu chưa thấy → chưa git push bản mới)
!grep -q "focal-gamma" VITASA_Enhanced/train_pair.py && echo "✅ train_pair.py có --focal-gamma (bản mới)" || echo "❌ CHƯA có --focal-gamma — git push bản mới trước!"
!ls VITASA_Enhanced/ | grep -E 'train_pair|text_norm|imbalanced'

In [ ]:
# 4b. Mount Google Drive — bắt buộc (nguồn lưu persistent + resume)
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/VITASA_Enhanced_results_backup"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"✅ Backup + resume qua: {DRIVE_BACKUP_DIR}")

In [ ]:
# 5. Smoke test — dùng --output riêng, KHÔNG đụng vào thư mục kết quả thật
# (bug đã gặp: nếu domain+flags trùng với 1 biến thể tuning thật mà thiếu
# --output, kết quả smoke test 1-epoch sẽ "giả làm" kết quả thật và bị
# resume-skip nhận nhầm là đã xong, không train lại nữa)
%cd VITASA_Enhanced
!python3 train_pair.py --domain mobile --loss focal --focal-gamma 1.0 --model phobert --epochs 1 --subsample 0.1 --batch-size 64 --fp16 --output /tmp/smoke_test_output 2>&1 | tail -25
print("\n✅ Smoke test done — nếu không lỗi, chạy tuning ở cell dưới")

In [ ]:
# 6. TUNING RUN — vòng 1 (3 biến thể) + vòng 2 (kết hợp), resume-safe qua Drive + git
import subprocess, time, shutil
from pathlib import Path

assert 'DRIVE_BACKUP_DIR' in dir(), "❌ Chạy cell '4b. Mount Google Drive' trước."

LOCAL_RESULTS_DIR = Path("experiments/results_pair")
DRIVE_DIR = Path(DRIVE_BACKUP_DIR) / "results_pair"
EPOCHS = 10           # GIỮ 10 epoch như vòng đầu để so sánh công bằng với C1-C4
BATCH_SIZE = 64

# (nhãn, flags, tên_config_dir) — tên phải khớp cách train_pair.py đặt (đã kiểm tra)
VARIANTS = [
    # --- vòng 1: test riêng lẻ từng giả thuyết ---
    ("focal_g1",       "--loss focal --focal-gamma 1.0",           "loss-focal_g1"),
    ("focal_noalpha",  "--loss focal --focal-no-alpha",            "loss-focal_noalpha"),
    ("norm_keepexpr",  "--loss ce --normalize --keep-expressive",  "loss-ce_norm_keepexpr"),
    # --- vòng 2: kết hợp các hướng đã có tín hiệu tốt ở restaurant/hotel ---
    # H1+H2: giảm gamma VÀ bỏ alpha cùng lúc (2 cơ chế làm dịu focal)
    ("focal_relaxed",  "--loss focal --focal-gamma 1.0 --focal-no-alpha",
                        "loss-focal_g1_noalpha"),
    # Full Model v2: gộp cả 3 điều chỉnh (focal dịu + giữ expressive marker)
    ("full_v2",        "--loss focal --focal-gamma 1.0 --focal-no-alpha --normalize --keep-expressive",
                        "loss-focal_norm_keepexpr_g1_noalpha"),
]
DOMAINS = ['mobile', 'restaurant', 'hotel']   # rẻ → đắt

def cfg_dir(domain, tag):
    return f"pair_{domain}_{tag}_phobert_mha"

failed = []
for domain in DOMAINS:
    for label, flags, tag in VARIANTS:
        config_name = cfg_dir(domain, tag)
        drive_rf = DRIVE_DIR / config_name / "results.json"
        git_rf   = LOCAL_RESULTS_DIR / config_name / "results.json"

        print(f"\n{'='*70}\n[{domain}/{label}] — {EPOCHS} epochs\n{'='*70}")
        if drive_rf.exists():
            print(f"⏭️  Đã có trên Drive ({drive_rf}) — bỏ qua."); continue
        if git_rf.exists():
            print(f"⏭️  Đã có sau git pull ({git_rf}) — bỏ qua + backup vào Drive.")
            drive_rf.parent.mkdir(parents=True, exist_ok=True); shutil.copy(git_rf, drive_rf); continue

        cmd = f"python3 train_pair.py --domain {domain} {flags} --model phobert --epochs {EPOCHS} --batch-size {BATCH_SIZE} --fp16"
        print(f"Command: {cmd}\n")
        r = subprocess.run(cmd.split(), capture_output=False)
        if r.returncode != 0:
            failed.append(f"{domain}/{label}"); print(f"❌ {domain}/{label} failed"); continue
        if git_rf.exists():
            drive_rf.parent.mkdir(parents=True, exist_ok=True); shutil.copy(git_rf, drive_rf)
            print(f"💾 Backed up: {drive_rf}")
        else:
            print(f"⚠️  Không thấy {git_rf} — kiểm tra tên config (chạy !ls experiments/results_pair)")
        time.sleep(3)

print(f"\n{'='*70}\nTuning done. Failed: {failed}\n{'='*70}")

In [ ]:
# 7. Bảng so sánh — CHỌN CẤU HÌNH THEO DEV, chỉ báo cáo TEST của cấu hình được chọn
# (tránh test set leakage: không được chọn cấu hình dựa trên điểm test)
import json
from pathlib import Path

assert 'DRIVE_BACKUP_DIR' in dir(), "❌ Chạy cell '4b. Mount Google Drive' trước."
DRIVE_DIR = Path(DRIVE_BACKUP_DIR) / "results_pair"

def load(config_name):
    f = DRIVE_DIR / config_name / "results.json"
    if not f.exists(): return None
    return json.load(open(f))

BASELINE = {"mobile": 61.77, "restaurant": 41.12, "hotel": 52.64}

VARIANTS = [
    ("C1",             "loss-ce"),
    ("C2 norm",        "loss-ce_norm"),
    ("C3 focal(g2)",   "loss-focal"),
    ("C4 full",        "loss-focal_norm"),
    ("focal_g1",       "loss-focal_g1"),
    ("focal_noalpha",  "loss-focal_noalpha"),
    ("norm_keepexpr",  "loss-ce_norm_keepexpr"),
    ("focal_relaxed",  "loss-focal_g1_noalpha"),
    ("full_v2",        "loss-focal_norm_keepexpr_g1_noalpha"),
]

print("="*100)
print("BƯỚC 1 — Bảng đầy đủ DEV vs TEST mọi biến thể (chỉ để tham khảo, KHÔNG dùng TEST để chọn)")
print("="*100)
for d in ['mobile','restaurant','hotel']:
    print(f"\n--- {d} ---")
    print(f"{'variant':16}{'DEV':>8}{'TEST':>8}")
    for name, tag in VARIANTS:
        r = load(f"pair_{d}_{tag}_phobert_mha")
        if r is None:
            continue
        print(f"{name:16}{r['best_dev_f1']*100:8.2f}{r['test']['macro_f1']*100:8.2f}")

print("\n" + "="*100)
print("BƯỚC 2 — CHỌN cấu hình theo DEV cao nhất, TEST chỉ nhìn đúng 1 lần cho cấu hình đó")
print("="*100)
print(f"{'domain':11}{'cấu hình chọn':16}{'dev':>8}{'TEST (số báo cáo)':>20}{'paper':>9}")
print("-"*100)
final_choice = {}
for d in ['mobile','restaurant','hotel']:
    best_dev, best_name, best_test = -1, None, None
    for name, tag in VARIANTS:
        r = load(f"pair_{d}_{tag}_phobert_mha")
        if r is None:
            continue
        if r["best_dev_f1"] > best_dev:
            best_dev = r["best_dev_f1"]
            best_name = name
            best_test = r["test"]["macro_f1"]
    final_choice[d] = (best_name, best_dev, best_test)
    print(f"{d:11}{best_name:16}{best_dev*100:8.2f}{best_test*100:20.2f}{BASELINE[d]:9.2f}")
print("="*100)
print("\n👉 Đây là số DUY NHẤT nên đưa vào báo cáo cho mỗi domain — không đổi lựa chọn")
print("   sau khi đã nhìn TEST. Nếu muốn thử cấu hình mới, phải coi đây là vòng mới,")
print("   không được quay lại sửa lựa chọn của vòng này dựa trên TEST vừa thấy.")


In [ ]:
# 8. Tải toàn bộ results về máy (gộp cả vòng-1 + tuning, từ Drive)
from google.colab import files
from pathlib import Path
assert 'DRIVE_BACKUP_DIR' in dir(), "❌ Chạy cell '4b. Mount Google Drive' trước."
results_dir = Path(DRIVE_BACKUP_DIR) / "results_pair"
!tar -czf VITASA_tuning_results.tar.gz -C "{results_dir.parent}" results_pair
!ls -lh VITASA_tuning_results.tar.gz
files.download("VITASA_tuning_results.tar.gz")
print("✅ Done")